In [1]:
import os, platform, torch

if platform.system() == 'Darwin':
    REPO_DIR = '/Users/amir/sciml/diffusion_data_assimilation'
    DATA_DIR = '/Users/amir/sciml/sea_ice_data'
else:
    REPO_DIR = '/home'
    DATA_DIR = '/mnt/sciml/a.sadreev/sea_ice_data'

os.chdir(REPO_DIR)
print(f"cwd: {os.getcwd()}")
print(f"data: {DATA_DIR}")
print(f"device: {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")

cwd: /Users/amir/sciml/diffusion_data_assimilation
data: /Users/amir/sciml/sea_ice_data
device: mps


In [3]:
import logging
import torch
from functools import partial
from trainer import TrainingConfig, UNetTrainer
from utils import NpyImageDataset, channel_normalize, add_noise
from diffusers.models.unets.unet_2d import UNet2DModel
from diffusers.optimization import get_cosine_schedule_with_warmup

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")

/Users/amir/sciml/diffusion_data_assimilation/.venv/lib/python3.14/site-packages/diffusers/models/transformers/transformer_kandinsky.py:168: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
/Users/amir/sciml/diffusion_data_assimilation/.venv/lib/python3.14/site-packages/diffusers/models/transformers/transformer_kandinsky.py:272: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)


In [ ]:
config = TrainingConfig()

transform = partial(channel_normalize, channel_mean=config.channel_mean, channel_std=config.channel_std)

# pin_memory ускоряет передачу данных только на CUDA; на MPS и CPU не нужен
use_pin_memory = torch.cuda.is_available()

dataset_train = NpyImageDataset(
    folder=f"{DATA_DIR}/valid",
    transform=transform,
    preload=False,
    mmap_mode='r',
)
train_dataloader = torch.utils.data.DataLoader(
    dataset_train,
    batch_size=config.train_batch_size,
    shuffle=True,
    num_workers=config.num_workers_train,
    pin_memory=use_pin_memory,
)

dataset_valid = NpyImageDataset(
    folder=f"{DATA_DIR}/test",
    transform=transform,
    preload=False,
    mmap_mode='r',
)
valid_dataloader = torch.utils.data.DataLoader(
    dataset_valid,
    batch_size=config.eval_batch_size,
    shuffle=False,
    num_workers=config.num_workers_val,
    pin_memory=use_pin_memory,
)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/amir/sciml/sea_ice_data/test'

In [ ]:
model = UNet2DModel(
    sample_size=config.image_size,
    in_channels=config.in_channels,
    out_channels=config.out_channels,
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 512, 512),
    down_block_types=(
        "DownBlock2D", "DownBlock2D", "DownBlock2D",
        "AttnDownBlock2D", "DownBlock2D",
    ),
    up_block_types=(
        "UpBlock2D", "AttnUpBlock2D", "UpBlock2D",
        "UpBlock2D", "UpBlock2D",
    ),
)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=config.lr_warmup_steps,
    num_training_steps=len(train_dataloader) * config.num_epochs,
)

trainer = UNetTrainer(
    config=config,
    model=model,
    optimizer=optimizer,
    data_loader_train=train_dataloader,
    data_loader_val=valid_dataloader,
    lr_scheduler=lr_scheduler,
    add_noise_func=add_noise,
)

In [ ]:
trainer.train_loop()

  0%|          | 0/365 [00:00<?, ?it/s]

Валидация на эпохе 0...
🏆 Новый лучший val_loss: 0.180662
Новая лучшая модель сохранена с val_loss: 0.180662


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Модель загружена на Hub для эпохи 0


  0%|          | 0/365 [00:00<?, ?it/s]

Валидация на эпохе 1...


KeyboardInterrupt: 